# Etapa 1 — Validar persistência Delta
O notebook cria apenas o schema de diagnóstico `medallion_dev_setup` no catálogo informado e a tabela `teste_delta`. Não remove objetos.
Valida criação de tabela, MERGE, atualização, releitura e repetição sem duplicidade. Isto ainda não comprova a idempotência do pipeline completo.
Ao concluir, compartilhe a saída da última célula. Não compartilhe credenciais.

**Parâmetros:** execute apenas a primeira célula de código para exibir os widgets. Preencha os campos e depois execute as demais células. Use **Run all** quando os parâmetros já estiverem preenchidos.


In [ ]:
# catalogo: preencha workspace (obrigatório). Define onde criar/consultar a tabela de diagnóstico.
# Nesta rodada, mantenha workspace para comparar com a validação anterior.
# Execute somente este CMD, preencha os widgets e então prossiga.
# 1. Exibir parâmetros — execute esta célula antes de preencher os campos.
dbutils.widgets.text("catalogo", spark.catalog.currentCatalog())

print("Preencha os widgets acima e depois execute as próximas células.")


In [ ]:
# CMD 2 — Ler catalogo, validar seu formato e criar/verificar schema e tabela Delta. Nenhum preenchimento adicional.
import re

catalogo = dbutils.widgets.get("catalogo").strip()
if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalogo):
    raise ValueError("Informe um catálogo com letras, números e sublinhado.")

schema = f"`{catalogo}`.`medallion_dev_setup`"
tabela = f"{schema}.`teste_delta`"
print(f"Spark: {spark.version}")
print(f"Destino do teste: {tabela}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {tabela} (
        id BIGINT,
        descricao STRING
    ) USING DELTA
""")
formato = spark.sql(f"DESCRIBE DETAIL {tabela}").first()["format"]
assert formato.lower() == "delta", f"Formato inesperado: {formato}"



In [ ]:
# CMD 3 — Executar MERGEs e conferir uma única linha id=1 atualizada. Esperado: VALIDAÇÃO DELTA: OK.
def gravar(descricao):
    # Valores literais controlados exclusivamente por este notebook.
    if descricao not in ("inserido", "atualizado"):
        raise ValueError("Descrição de teste inválida")
    spark.sql(f"""
        MERGE INTO {tabela} AS destino
        USING (SELECT CAST(1 AS BIGINT) AS id, '{descricao}' AS descricao) AS origem
        ON destino.id = origem.id
        WHEN MATCHED THEN UPDATE SET descricao = origem.descricao
        WHEN NOT MATCHED THEN INSERT (id, descricao)
        VALUES (origem.id, origem.descricao)
    """)

gravar("inserido")
gravar("atualizado")
gravar("atualizado")

resultado = spark.table(tabela).where("id = 1").collect()
assert len(resultado) == 1, f"Esperada uma linha; encontradas {len(resultado)}"
assert resultado[0]["descricao"] == "atualizado", "Atualização não persistida"

print("VALIDAÇÃO DELTA: OK")
print(f"Catálogo: {catalogo}")
print("Schema: medallion_dev_setup")
print("Formato: delta")
print("MERGE e releitura: OK")
print("Repetição sem duplicidade para id=1: OK")



## Coleta desta revalidação

Após executar todos os CMDs, confira a mensagem final esperada. Se ocorrer erro, preserve o output e interrompa a sequência.

Exporte este notebook como **IPYNB com outputs**, abra o arquivo e confirme que os resultados estão presentes. Salve o arquivo antes de limpar outputs ou mudar a sessão. Faça commit/push na branch `chore/estrutura-inicial`, registre o SHA e envie o export para revisão antes de avançar ao próximo notebook.

Sugestão de commit: `docs: revalida 00_validar_delta com instrucoes de parametros`.

O commit sozinho não comprova que os outputs foram incluídos. A cópia pública do export será revisada para remover e-mails e caminhos pessoais, com manifesto e hashes, preservando a coleta anterior. Esta rodada só será registrada como validada após revisão do novo export.
